

### Ziel dieser Datei
Benötigte Rohdaten für gesamte Schweiz runterladen. Dann filtern auf die relevanten Attribute (Wiese, Wälder, Schutzzonen usw). Dann auch sicherstellen, dass alle Datensätze in LV95 sind.

### Datenquellen
OSM-Daten über OverpassTurbo die jeweiligen Daten holen. Korrekte Abfragefilter machen, damit die richtigen Daten heruntergeladen werden!
  * Wiesen/Wälder (z.B. `landuse=meadow`, `landuse=forest`, usw.)
  * Infrastruktur (z.B. `amenity=farm`, `landuse=farmyard` für Bauernhöfe; `highway=bus_stop`, `railway=station` für ÖV)
  * Wenn möglich Geländeneigung, sonst mindestens natürliche Gefahrenzonen (`natural=cliff` für Felswände, `natural=scree` für Geröll) um dann diese Gebiete später auszuschliessen
Schutzgebiete Vektordatensatz als GeoJSON/Geopackage von geo.admin.ch herunterladen.

### Grober Codeaufbau
1. GeoJSON-Dateien einlesen
2. Koordinatensystem anpassen in LV95 (für berechnungen / verschnitte später)
3. Attributtabellen bereinigen: Unnötige Spalten löschen, damit die Dateien performant laufen.

### Export & Übernahme für die Nächste Datei 2
* Bereinigte Daten einzeln als GeoJSON abspeichern!
* Saubere Benennung nach Thematik, dass für Import in Datei 2 alles klar ist. z.B. `bearbeitet_flächen.geojson` / `bearbeitet_schutzgebiete.geojson`.

In [5]:
import osmnx as ox
import geopandas as gpd
import warnings
import pandas as pd

# Warnungen unterdrücken für eine saubere Ausgabe
warnings.filterwarnings('ignore')

# 1. OSM-Daten beziehen
place_name = "Kanton Basel-Landschaft, Switzerland"

print("Lade Wälder und Wiesen...")
tags_nature = {'landuse': ['meadow', 'forest']}
gdf_nature = ox.features_from_place(place_name, tags_nature)

print("Lade Infrastruktur (Bauernhöfe, ÖV)...")
tags_infra = {
    'amenity': ['farm'],
    'landuse': ['farmyard'],
    'highway': ['bus_stop'],
    'railway': ['station'],
    'emergency': ['fire_hydrant']
}
gdf_infra = ox.features_from_place(place_name, tags_infra)

print("Lade Hydrantenstandorte herunter...")
tags_hydrants = {'emergency': 'fire_hydrant'}
gdf_hydrants = ox.features_from_place(place_name, tags_hydrants)

print("Lade Gefahrenzonen...")
tags_hazards = {'natural': ['cliff', 'scree']}
gdf_hazards = ox.features_from_place(place_name, tags_hazards)

# ==========================================
# 2. Schutzgebiete einlesen (Shapefiles & GPKG)
# ==========================================
print("Lade Schutzgebiete (Lokale Dateien)...")
# Pfade gemäss deiner VS-Code Ordnerstruktur
pfad_jagdbann = 'data_BL/bundesinventare-jagdbanngebiete_2056.shp/N2023_Revision_jagdbann.shp'
pfad_moor = 'data_BL/bundesinventare-moorlandschaften_2056.shp/N2017_Revision_Moorlandschaft_20171101.shp'
pfad_auen = 'data_BL/auen-vegetationskarten_2056.gpkg'
pfad_wohngeb = 'data_BL/wohngebiete-aulav_2056.gpkg'

gdf_jagdbann = gpd.read_file(pfad_jagdbann)
gdf_moor = gpd.read_file(pfad_moor)
gdf_auen = gpd.read_file(pfad_auen, layer='Auenvegetation')
gdf_wohngeb = gpd.read_file(pfad_wohngeb, layer='Bufferzone')

# ==========================================
# 3. Koordinatensystem in LV95 (EPSG:2056) sicherstellen/transformieren
# ==========================================
print("Transformiere in LV95...")
gdf_nature_lv95 = gdf_nature.to_crs(epsg=2056)
gdf_infra_lv95 = gdf_infra.to_crs(epsg=2056)
gdf_hazards_lv95 = gdf_hazards.to_crs(epsg=2056)
gdf_hydrants_lv95 = gdf_hydrants.to_crs(epsg=2056)

# Die lokalen Dateien haben zwar _2056 im Namen, wir stellen aber zur Sicherheit sicher, 
# dass der CRS-Metadaten-Tag in Python korrekt auf 2056 gesetzt ist.
gdf_jagdbann_lv95 = gdf_jagdbann.to_crs(epsg=2056)
gdf_moor_lv95 = gdf_moor.to_crs(epsg=2056)
gdf_auen_lv95 = gdf_auen.to_crs(epsg=2056)
gdf_wohngeb_lv95 = gdf_wohngeb.to_crs(epsg=2056)

# ==========================================
# 4. Attributtabellen bereinigen & kombinieren
# ==========================================
def clean_attributes(gdf, keep_columns):
    existing_cols = [col for col in keep_columns if col in gdf.columns] + ['geometry']
    return gdf[existing_cols]

print("Bereinige Attributtabellen...")
gdf_nature_clean = clean_attributes(gdf_nature_lv95, ['landuse'])
gdf_infra_clean = clean_attributes(gdf_infra_lv95, ['amenity', 'landuse', 'highway', 'railway', 'name'])
gdf_hazards_clean = clean_attributes(gdf_hazards_lv95, ['natural'])
gdf_hydrants_clean = clean_attributes(gdf_hydrants_lv95, ['emergency'])

# Infrastruktur & Hydranten zusammenfügen
gdf_infrastructure_clean = pd.concat([gdf_infra_clean, gdf_hydrants_clean], ignore_index=True)

# SCHUTZGEBIETE ZUSAMMENFÜGEN:
# Da wir diese später nur zum "Ausschliessen" (Wegschneiden) brauchen, 
# behalten wir nur die Geometrie und geben ihnen eine einfache Typ-Bezeichnung.
gdf_jagdbann_clean = clean_attributes(gdf_jagdbann_lv95, [])
gdf_jagdbann_clean['schutz_typ'] = 'Jagdbanngebiet'

gdf_moor_clean = clean_attributes(gdf_moor_lv95, [])
gdf_moor_clean['schutz_typ'] = 'Moorlandschaft'

gdf_auen_clean = clean_attributes(gdf_auen_lv95, [])
gdf_auen_clean['schutz_typ'] = 'Aue'

gdf_wohngeb_clean = clean_attributes(gdf_wohngeb_lv95, [])
gdf_wohngeb_clean['schutz_typ'] = 'Siedlung'

# Alle drei Schutzgebiete zu einem einzigen Datensatz zusammenfügen
gdf_schutz_kombiniert = pd.concat([gdf_jagdbann_clean, gdf_moor_clean, gdf_auen_clean, gdf_wohngeb_clean], ignore_index=True)
# Wichtig nach einem pd.concat mit Geodaten: Wieder offiziell als GeoDataFrame deklarieren
gdf_schutz_kombiniert = gpd.GeoDataFrame(gdf_schutz_kombiniert, geometry='geometry', crs="EPSG:2056")


# ==========================================
# 5. Export als bereinigte GeoJSON-Dateien für Datei 2
# ==========================================
print("Exportiere GeoJSON-Dateien...")
gdf_nature_clean.to_file("data_BL/bearbeitet_flaechen.geojson", driver="GeoJSON")
gdf_infrastructure_clean.to_file("data_BL/bearbeitet_infrastruktur.geojson", driver="GeoJSON")
gdf_hazards_clean.to_file("data_BL/bearbeitet_gefahrenzonen.geojson", driver="GeoJSON")

# Optional: Wenn du die Hydranten doch noch einzeln brauchst, kannst du sie auch speichern:
# gdf_hydrants_clean.to_file("data_BL/bearbeitet_hydranten.geojson", driver="GeoJSON")

# Export der kombinierten Schutzgebiete
gdf_schutz_kombiniert.to_file("data_BL/bearbeitet_schutzgebiete.geojson", driver="GeoJSON")

print("Datenbezug und Export erfolgreich abgeschlossen!")

Lade Wälder und Wiesen...
Lade Infrastruktur (Bauernhöfe, ÖV)...
Lade Hydrantenstandorte herunter...
Lade Gefahrenzonen...
Lade Schutzgebiete (Lokale Dateien)...
Transformiere in LV95...
Bereinige Attributtabellen...
Exportiere GeoJSON-Dateien...
Datenbezug und Export erfolgreich abgeschlossen!
